In [1]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml.feature import VectorAssembler
from sklearn.metrics import roc_curve, auc, precision_recall_curve
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt



class RandomForestModel:
    """
    Random Forest Model for Loan Prediction
    """

    def __init__(self):
        self.model = None
        self.feature_names = [
            "Age", "Income", "LoanAmount", "Credit_Score",
            "Employment_Years", "Credit_History", "Has_Defaulted",
            "Dependents", "Gender", "Education_Level",
            "Married", "Job_Type", "Property_Area"
        ]

    def build_features(self, df):
        """
        Build feature vector from raw DataFrame
        """
        # Rename target column
        df = df.withColumnRenamed("Loan_Status", "label")

        # Define feature columns
        feature_cols = [
            "Age", "Income", "LoanAmount", "Credit_Score",
            "Employment_Years", "Credit_History", "Has_Defaulted",
            "Dependents"
        ]
        categorical_cols = ["Gender", "Education_Level", "Married", "Job_Type", "Property_Area"]

        # Combine all features
        all_features = feature_cols + categorical_cols

        # Create vector assembler
        assembler = VectorAssembler(
            inputCols=all_features,
            outputCol="features",
            handleInvalid="skip"
        )

        # Transform DataFrame
        df = assembler.transform(df)
        df = df.dropna()

        print("Features ready")
        print(f"Total rows: {df.count()}")

        return df

    def split_data(self, df, ratio=0.8):
        """
        Split data into train and test sets
        """
        train_df, test_df = df.randomSplit([ratio, 1-ratio], seed=42)
        print(f"Training data: {train_df.count()} rows")
        print(f"Testing data: {test_df.count()} rows")
        return train_df, test_df

    def create_model(self):
        """
        Create Random Forest classifier with optimized parameters
        """
        return RandomForestClassifier(
            featuresCol="features",
            labelCol="label",
            predictionCol="prediction",
            probabilityCol="probability",
            numTrees=200,
            maxDepth=15,
            minInstancesPerNode=10,
            minInfoGain=0.01,
            impurity="gini",
            subsamplingRate=0.8,
            seed=42,
            cacheNodeIds=True,
            maxBins=32
        )

    def train_model(self, train_df):
        """
        Train Random Forest model
        """
        rf = self.create_model()
        print("Training Random Forest...")
        self.model = rf.fit(train_df)
        print("Training complete!")
        return self.model

    def predict(self, test_df):
        """
        Make predictions using trained model
        """
        print("Making predictions...")
        predictions = self.model.transform(test_df)
        print(" Predictions complete")
        return predictions

    def evaluate(self, predictions):
        """
        Calculate all evaluation metrics
        """
        # Accuracy
        acc = MulticlassClassificationEvaluator(
            labelCol="label", predictionCol="prediction", metricName="accuracy"
        ).evaluate(predictions)

        # F1 Score
        f1 = MulticlassClassificationEvaluator(
            labelCol="label", predictionCol="prediction", metricName="f1"
        ).evaluate(predictions)

        # AUC
        auc_score = BinaryClassificationEvaluator(
            labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
        ).evaluate(predictions)

        # Precision
        precision = MulticlassClassificationEvaluator(
            labelCol="label", predictionCol="prediction", metricName="precisionByLabel", metricLabel=1.0
        ).evaluate(predictions)

        # Recall
        recall = MulticlassClassificationEvaluator(
            labelCol="label", predictionCol="prediction", metricName="recallByLabel", metricLabel=1.0
        ).evaluate(predictions)

        return acc, f1, auc_score, precision, recall

    def print_evaluation(self, predictions):
        """
        Print all evaluation metrics
        """
        acc, f1, auc_score, precision, recall = self.evaluate(predictions)
        print(" RANDOM FOREST - EVALUATION RESULTS")
        print(f"Accuracy:   {acc:.4f}")
        print(f"F1 Score:   {f1:.4f}")
        print(f"AUC:        {auc_score:.4f}")
        print(f"Precision:  {precision:.4f}")
        print(f"Recall:     {recall:.4f}")

        return acc, f1, auc_score, precision, recall


    def plot_confusion_matrix(self, predictions):
        """
        Plot confusion matrix heatmap
        """
        pred_pd = predictions.select("label", "prediction").toPandas()
        cm = pd.crosstab(pred_pd["label"], pred_pd["prediction"])

        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Greens")
        plt.title("Random Forest - Confusion Matrix", fontsize=14, fontweight='bold')
        plt.xlabel("Predicted", fontsize=12)
        plt.ylabel("Actual", fontsize=12)
        plt.tight_layout()
        plt.show()

        return cm

    def plot_roc_curve(self, predictions):
        """
        Plot ROC curve
        """
        pred_pd = predictions.select("label", "probability").toPandas()
        y_true = pred_pd["label"]
        y_score = pred_pd["probability"].apply(lambda x: float(x[1]))

        fpr, tpr, _ = roc_curve(y_true, y_score)
        roc_auc = auc(fpr, tpr)

        plt.figure(figsize=(8, 6))
        plt.plot(fpr, tpr, 'g-', lw=2, label=f'Random Forest (AUC = {roc_auc:.3f})')
        plt.plot([0, 1], [0, 1], 'r--', lw=2, label='Random Classifier')
        plt.xlabel("False Positive Rate (FPR)", fontsize=12)
        plt.ylabel("True Positive Rate (TPR)", fontsize=12)
        plt.title("ROC Curve - Random Forest", fontsize=14, fontweight='bold')
        plt.legend(loc='lower right')
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

        return roc_auc

    def plot_precision_recall_curve(self, predictions):
        """
        Plot Precision-Recall curve
        """
        pred_pd = predictions.select("label", "probability").toPandas()
        y_true = pred_pd["label"]
        y_score = pred_pd["probability"].apply(lambda x: float(x[1]))

        precision_curve, recall_curve, _ = precision_recall_curve(y_true, y_score)

        plt.figure(figsize=(8, 6))
        plt.plot(recall_curve, precision_curve, 'g-', linewidth=2)
        plt.xlabel("Recall", fontsize=12)
        plt.ylabel("Precision", fontsize=12)
        plt.title("Precision-Recall Curve - Random Forest", fontsize=14)
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

    def plot_feature_importance(self):
        """
        Plot feature importance bar chart
        """
        importance = self.model.featureImportances.toArray()

        fi_df = pd.DataFrame({
            "Feature": self.feature_names,
            "Importance": importance
        }).sort_values("Importance", ascending=False)

        plt.figure(figsize=(10, 8))
        sns.barplot(x="Importance", y="Feature", data=fi_df, palette="viridis")
        plt.title("Random Forest - Feature Importance", fontsize=14, fontweight='bold')
        plt.xlabel("Importance Score", fontsize=12)
        plt.tight_layout()
        plt.show()

        # Print ranking
        print("\nFeature Importance Ranking (Random Forest):")
        print("="*45)
        for i, row in fi_df.iterrows():
            print(f"{row['Feature']:20} : {row['Importance']:.4f}")

        return fi_df

    def plot_performance_metrics(self, predictions):
        """
        Plot all performance metrics as bar chart
        """
        acc, f1, auc_score, precision, recall = self.evaluate(predictions)

        metrics = ['Accuracy', 'F1 Score', 'AUC', 'Precision', 'Recall']
        scores = [acc, f1, auc_score, precision, recall]
        colors = ['#2ecc71', '#27ae60', '#229954', '#1e8449', '#196f3d']

        plt.figure(figsize=(10, 6))
        bars = plt.bar(metrics, scores, color=colors)
        plt.ylim(0, 1)
        plt.ylabel('Score', fontsize=12)
        plt.title('Random Forest - Performance Metrics', fontsize=14, fontweight='bold')

        # Add values on top of bars
        for bar, score in zip(bars, scores):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f'{score:.3f}', ha='center', fontsize=10)

        plt.tight_layout()
        plt.show()

    def save_model(self, path="random_forest_model"):
        """
        Save trained model
        """
        if self.model:
            self.model.save(path)
            print(f" Model saved to {path}")
        else:
            print("No model to save. Train the model first.")

    def get_feature_importance_df(self):
        """
        Get feature importance as DataFrame
        """
        importance = self.model.featureImportances.toArray()
        fi_df = pd.DataFrame({
            "Feature": self.feature_names,
            "Importance": importance
        }).sort_values("Importance", ascending=False)
        return fi_df


"""
# Initialize Random Forest Model
rf_model = RandomForestModel()

# Build features
df_ready = rf_model.build_features(df)

# Split data
train_df, test_df = rf_model.split_data(df_ready)

# Train model
rf_model.train_model(train_df)

# Make predictions
predictions = rf_model.predict(test_df)

# Evaluate and print results
rf_model.print_evaluation(predictions)

# Visualizations
rf_model.plot_confusion_matrix(predictions)
rf_model.plot_roc_curve(predictions)
rf_model.plot_precision_recall_curve(predictions)
rf_model.plot_feature_importance()
rf_model.plot_performance_metrics(predictions)

# Save model
rf_model.save_model()
"""

'\n# Initialize Random Forest Model\nrf_model = RandomForestModel()\n\n# Build features\ndf_ready = rf_model.build_features(df)\n\n# Split data\ntrain_df, test_df = rf_model.split_data(df_ready)\n\n# Train model\nrf_model.train_model(train_df)\n\n# Make predictions\npredictions = rf_model.predict(test_df)\n\n# Evaluate and print results\nrf_model.print_evaluation(predictions)\n\n# Visualizations\nrf_model.plot_confusion_matrix(predictions)\nrf_model.plot_roc_curve(predictions)\nrf_model.plot_precision_recall_curve(predictions)\nrf_model.plot_feature_importance()\nrf_model.plot_performance_metrics(predictions)\n\n# Save model\nrf_model.save_model()\n'